# **Spacy Finetuning**

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%cd /content/drive/MyDrive/NER


/content/drive/MyDrive/NER


In [10]:
import pandas as pd
import torch
import spacy
import pandas as pd
from difflib import SequenceMatcher
from spacy.tokens import DocBin
import os
import json
from spacy.training import Example


In [11]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'
print(device)

cuda


## Init and train Armenian Spacy

In [12]:
import random
from pathlib import Path
from spacy.util import minibatch
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.scorer import Scorer
from tqdm.notebook import tqdm   # for Jupyter-friendly progress bars



nlp = spacy.blank("hy")
ner = nlp.add_pipe("ner")

ner.add_label("INSC")


# ---------------------------
# Load validation data
# ---------------------------
dev_docbin = DocBin().from_disk("spacy_data/dev.spacy")
dev_docs = list(dev_docbin.get_docs(nlp.vocab))

def evaluate(nlp, docs):
    """Evaluate model on validation docs."""
    examples = []
    for gold_doc in docs:
        pred_doc = nlp(gold_doc.text)
        examples.append(Example(pred_doc, gold_doc))
    scorer = Scorer()
    return scorer.score(examples)


# Load train data
train_docbin = DocBin().from_disk("spacy_data/baseline.spacy")
train_docs = list(train_docbin.get_docs(nlp.vocab))

train_examples = []
for doc in train_docs:
    ents = [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]
    train_examples.append(Example.from_dict(nlp.make_doc(doc.text), {"entities": ents}))

# Initialize model
optimizer = nlp.initialize(lambda: train_examples)

# ---------------------------
# Training Hyperparameters
# ---------------------------
n_iter = 20
patience = 3
patience_counter = 0
best_f1 = 0.0
best_model_path = Path("models/best_model")
best_model_path.mkdir(parents=True, exist_ok=True)

# ---------------------------
# Training Loop
# ---------------------------
for epoch in range(n_iter):
    random.shuffle(train_examples)
    losses = {}

    # tqdm progress bar over batches
    batches = list(minibatch(train_examples, size=16))
    progress_bar = tqdm(batches, desc=f"Epoch {epoch+1}/{n_iter}")

    for batch in progress_bar:
        nlp.update(batch, losses=losses, sgd=optimizer)
        progress_bar.set_postfix(loss=losses.get("ner", 0))

    # ---- VALIDATION ----
    val_scores = evaluate(nlp, dev_docs)
    f1 = val_scores["ents_f"]
    p  = val_scores["ents_p"]
    r  = val_scores["ents_r"]

    print(f"\nEpoch {epoch+1}/{n_iter}")
    print(f"  Train Loss: {losses}")
    print(f"  Val F1: {f1:.4f}, P: {p:.4f}, R: {r:.4f}")

    # ---- EARLY STOPPING + BEST MODEL SAVING ----
    if f1 > best_f1:
        best_f1 = f1
        patience_counter = 0
        nlp.to_disk(best_model_path)
        print("  🌟 New best model saved!")
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("  🔥 Early stopping triggered!")
            break


Epoch 1/20:   0%|          | 0/76 [00:00<?, ?it/s]


Epoch 1/20
  Train Loss: {'ner': np.float32(1197.8363)}
  Val F1: 0.8148, P: 1.0000, R: 0.6875
  🌟 New best model saved!


Epoch 2/20:   0%|          | 0/76 [00:00<?, ?it/s]


Epoch 2/20
  Train Loss: {'ner': np.float32(77.26543)}
  Val F1: 0.5833, P: 0.8750, R: 0.4375
  No improvement. Patience: 1/3


Epoch 3/20:   0%|          | 0/76 [00:00<?, ?it/s]

...


Epoch 7/20
  Train Loss: {'ner': np.float32(9.854514)}
  Val F1: 0.9032, P: 0.9333, R: 0.8750
  No improvement. Patience: 1/3


Epoch 8/20:   0%|          | 0/76 [00:00<?, ?it/s]


Epoch 8/20
  Train Loss: {'ner': np.float32(10.898938)}
  Val F1: 0.9032, P: 0.9333, R: 0.8750
  No improvement. Patience: 2/3


Epoch 9/20:   0%|          | 0/76 [00:00<?, ?it/s]


Epoch 9/20
  Train Loss: {'ner': np.float32(6.5089993)}
  Val F1: 0.8824, P: 0.8333, R: 0.9375
  No improvement. Patience: 3/3
  🔥 Early stopping triggered!


## Evaluating the Model

In [14]:
import spacy
from spacy.tokens import DocBin
from spacy.training import Example
from spacy.scorer import Scorer

# ---- Load best model ----
nlp_best = spacy.load("models/best_model")

# ---- Load test set ----
test_docbin = DocBin().from_disk("spacy_data/test.spacy")
test_docs = list(test_docbin.get_docs(nlp_best.vocab))

def evaluate(nlp, docs):
    examples = []
    for gold_doc in docs:
        pred_doc = nlp(gold_doc.text)
        examples.append(Example(pred_doc, gold_doc))
    scorer = Scorer()
    return scorer.score(examples)

# ---- Run evaluation ----
results = evaluate(nlp_best, test_docs)

print("=== TEST SET PERFORMANCE ===")
print(f"Precision: {results['ents_p']:.4f}")
print(f"Recall:    {results['ents_r']:.4f}")
print(f"F1-score:  {results['ents_f']:.4f}")


=== TEST SET PERFORMANCE ===
Precision: 0.5000
Recall:    0.0588
F1-score:  0.1053


### Diagnosis

In [15]:
from spacy.training.iob_utils import offsets_to_biluo_tags

# ---------------------------
# Misalignment Diagnostic
# ---------------------------
print("\n🔍 Checking annotation/tokenization alignment...")

bad = 0
total = 0

for doc in train_docs:
    text = doc.text
    # Convert gold spans into offsets
    entities = [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]

    # Check alignment against spaCy tokenization
    temp_doc = nlp.make_doc(text)
    tags = offsets_to_biluo_tags(temp_doc, entities)

    if "-" in tags:   # '-' = misaligned
        bad += 1
    total += 1

misalign_ratio = bad / total if total else 0

print(f"Misalignment: {bad}/{total} → {misalign_ratio:.4f}")

if misalign_ratio > 0.05:
    print("⚠️ WARNING: High misalignment! Your training quality will suffer.")
    print("   > Fix entity boundaries or tokenization to improve performance.")
else:
    print("✅ Good alignment! You can proceed safely.")



🔍 Checking annotation/tokenization alignment...
Misalignment: 0/1207 → 0.0000
✅ Good alignment! You can proceed safely.
